# GEMINI

In [ ]:
import os
!pip install -q -U google-generativeai
import google.generativeai as genai
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-1.5-pro')
response = model.generate_content('Who are you')
print(response.text)

In [ ]:
model = genai.GenerativeModel("gemini-1.5-flash")
chat = model.start_chat(
    history=[
        {"role": "user", "parts": "Hello"},
        {"role": "model", "parts": "Great to meet you. What would you like to know?"},
    ]
)
response = chat.send_message("I have 2 dogs in my house.")
print(response.text)
response = chat.send_message("How many paws are in my house?")
print(response.text)

In [ ]:
import os
import streamlit as st
import google.generativeai as genai

def main():
    st.title("Chat with Gemini")

    # Configure the API key
    GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
    genai.configure(api_key=GOOGLE_API_KEY)

    # Initialize the chat history in the session state if it doesn't exist
    if "chat_history" not in st.session_state:
        st.session_state.chat_history = [
            {"role": "user", "parts": "Hello"},
            {"role": "model", "parts": "Great to meet you. What would you like to know?"},
        ]

    # Display the chat history
    chat_display = st.empty()
    with chat_display.container():
        for message in st.session_state.chat_history:
            if message["role"] == "user":
                st.write(f"**User:** {message['parts']}")
            else:
                st.write(f"**Model:** {message['parts']}")

    # User input
    user_input = st.text_input("Enter your message:")
    if user_input:
        # Send the message and get a response
        model = genai.GenerativeModel("gemini-1.5-flash")
        chat = model.start_chat(history=st.session_state.chat_history)
        response = chat.send_message(user_input)

        # Update the session state with the new user message and model response
        st.session_state.chat_history.append({"role": "user", "parts": user_input})
        st.session_state.chat_history.append({"role": "model", "parts": response.text})

        # Clear and update the chat display
        chat_display.empty()
        with chat_display.container():
            for message in st.session_state.chat_history:
                if message["role"] == "user":
                    st.write(f"**User:** {message['parts']}")
                else:
                    st.write(f"**Model:** {message['parts']}")

if __name__ == "__main__":
    main()


In [ ]:
pip install streamlit

In [ ]:
import os
import streamlit as st
import google.generativeai as genai

# Replace with your actual API key
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

def generate_response(prompt):
  response = model.generate_content(prompt)
  return response.text

st.title("Gemini Model Demo")
user_input = st.text_input("Enter your prompt:")

if user_input:
  response = generate_response(user_input)
  st.write(response)

In [ ]:
import os
import streamlit as st
import google.generativeai as genai

# Replace with your actual API key
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")
chat = model.start_chat()

def generate_response(prompt):
    global chat
    try:
        response = chat.send_message(prompt)
        return response.text
    except Exception as e:
        st.error(f"Error generating response: {e}")
        return "An error occurred."

st.title("Gemini Model Demo")
user_input = st.text_input("Enter your prompt:")

if user_input:
    response = generate_response(user_input)
    st.write(response)

In [ ]:
!streamlit run response.py

In [ ]:
!pip install av

In [ ]:
import os
import requests

DEEPGRAM_URL = "https://api.deepgram.com/v1/speak?model=aura-asteria-en"
DEEPGRAM_API_KEY = os.environ["DEEPGRAM_API_KEY"]

payload = {
    "text": "Hello, how can I help you today? My name is Emily and I'm very glad to meet you. What do you think of this new text-to-speech API?"
}

headers = {
    "Authorization": f"Token {DEEPGRAM_API_KEY}",
    "Content-Type": "application/json"
}

audio_file_path = "output.mp3"  # Path to save the audio file

with open(audio_file_path, 'wb') as file_stream:
    response = requests.post(DEEPGRAM_URL, headers=headers, json=payload, stream=True)
    for chunk in response.iter_content(chunk_size=1024):
        if chunk:
            file_stream.write(chunk)





In [ ]:
pip install -q -U groq

# **WHISPER**

In [ ]:
import streamlit as st
from groq import Groq
from gtts import gTTS
import os

def reply(prompt, history=[]):
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)
    history.append({"role": "user", "content": prompt})

    chat_complete = client.chat.completions.create(
        messages=history,
        model='llama-3.1-70b-versatile',
        max_tokens=1024,
        temperature=0.5,
        stop=None,
        stream=None
    )

    history.append({"role": "assistant", "content": chat_complete.choices[0].message.content})
    if len(history) > 5:
        history = history[-5:]
    return chat_complete.choices[0].message.content

def transcribe_audio(uploaded_file, history=[]):
    """Transcribes audio using Groq's Whisper model and returns the text."""
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)

    try:
        # Access uploaded file content
        audio_content = uploaded_file.read()
        translation = client.audio.translations.create(
            file=(uploaded_file.name, audio_content), model="whisper-large-v3"
        )

        return reply(translation.text, history)
    except Exception as e:
        st.error(f"Error during transcription: {e}")
        return None

def convert_text_to_speech(text):
    """Convert text to speech using gTTS."""
    tts = gTTS(text, lang='en')
    audio_file = "output.mp3"
    tts.save(audio_file)
    return audio_file

def main():
    """Streamlit app to upload, transcribe, and convert text to speech."""
    st.title("Audio Transcription & Text Input with Groq & Text-to-Speech")

    # Initialize session state for history
    if "history" not in st.session_state:
        st.session_state.history = []

    response_text = None

    # Add a text input box for direct text input
    user_input = st.text_input("Enter text directly:")

    # Add file uploader for audio input
    uploaded_file = st.file_uploader("Or choose an audio file (WAV, FLAC, MP3)", type=["audio/wav", "audio/flac", "audio/mpeg", 'mp3', 'm4a', 'wav'])

    if uploaded_file is not None:
        with st.spinner("Transcribing..."):
            transcribed_text = transcribe_audio(uploaded_file, st.session_state.history)
            if transcribed_text:
                response_text = reply(transcribed_text, st.session_state.history)

    elif user_input:
        response_text = reply(user_input, st.session_state.history)

    if response_text:
        st.write("Response Text:")
        st.write(response_text)

        # Convert the response to speech
        audio_file = convert_text_to_speech(response_text)
        st.audio(audio_file, format='audio/mp3')

if __name__ == "__main__":
    main()


In [ ]:
pip install streamlit-webrtc

In [ ]:
from streamlit_webrtc import webrtc_streamer, AudioProcessorBase, ClientSettings
import numpy as np
import av
from scipy.io.wavfile import write
import tempfile

In [ ]:
!pip install streamlit-audio-recorder

In [ ]:
!pip install gtts


In [ ]:
!pip install pyaudio sounddevice numpy

In [ ]:
pip install streamlit-audio-recorder


In [ ]:
! pip install streamlit -q
!wget -q -O - ipv4.icanhazip.com
! streamlit run app.py & npx localtunnel --port 8501

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from groq import Groq
from gtts import gTTS
import os
import numpy as np
from scipy.io import wavfile

def reply(prompt, history=[]):
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)
    history.append({"role": "user", "content": prompt})

    chat_complete = client.chat.completions.create(
        messages=history,
        model='llama-3.3-70b-versatile',
        max_tokens=1024,
        temperature=0.5,
        stop=None,
        stream=None
    )

    response = chat_complete.choices[0].message.content
    history.append({"role": "assistant", "content": response})
    if len(history) > 5:
        history = history[-5:]
    return response, history

def transcribe_audio(audio):
    """Transcribes audio using Groq's Whisper model and returns the text."""
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)

    try:
        # Save the audio array as a temporary WAV file
        temp_filename = "temp_audio.wav"
        wavfile.write(temp_filename, rate=audio[0], data=audio[1])

        with open(temp_filename, "rb") as f:
            audio_content = f.read()

        translation = client.audio.translations.create(
            file=(temp_filename, audio_content), model="whisper-large-v3"
        )

        # Remove the temporary file
        os.remove(temp_filename)

        return translation.text
    except Exception as e:
        print(f"Error during transcription: {e}")
        return None

def convert_text_to_speech(text):
    """Convert text to speech using gTTS."""
    tts = gTTS(text, lang='en')
    audio_file = "output.mp3"
    tts.save(audio_file)
    return audio_file

def process_input(input_type, text_input, audio_input, history):
    if input_type == "text":
        response, new_history = reply(text_input, history)
    elif input_type == "audio":
        transcribed_text = transcribe_audio(audio_input)
        if transcribed_text:
            response, new_history = reply(transcribed_text, history)
        else:
            response = "Failed to transcribe audio."
            new_history = history
    else:
        response = "Invalid input type."
        new_history = history

    audio_response = convert_text_to_speech(response)
    return response, audio_response, new_history

def chat_interface(input_type, text_input, audio_input, history):
    response, audio_response, new_history = process_input(input_type, text_input, audio_input, history)
    return response, audio_response, new_history

# Define the Gradio interface
iface = gr.Interface(
    fn=chat_interface,
    inputs=[
        gr.Radio(["text", "audio"], label="Input Type"),
        gr.Textbox(label="Text Input"),
        gr.Audio(type="numpy", label="Audio Input"),
        gr.State([])  # For maintaining conversation history
    ],
    outputs=[
        gr.Textbox(label="Response"),
        gr.Audio(label="Audio Response"),
        gr.State()  # For updating conversation history
    ],
    title="Groq Chat Interface with Audio Support",
    description="Chat with the Groq AI model using text or audio input."
)

# Launch the interface
iface.launch()

# WAIT

In [ ]:
#!apt install libasound2-dev portaudio19-dev libportaudio2 libportaudiocpp0 ffmpeg
!pip install pyaudio

In [ ]:
pip install gradio

In [ ]:
import gradio as gr
from groq import Groq
from gtts import gTTS
import os
import numpy as np
from scipy.io import wavfile
import pyaudio
import wave
import threading
import queue

def reply(prompt, history=[]):
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)
    history.append({"role": "user", "content": prompt})

    chat_complete = client.chat.completions.create(
        messages=history,
        model='llama-3.3-70b-versatile',
        max_tokens=1024,
        temperature=0.5,
        stop=None,
        stream=None
    )

    response = chat_complete.choices[0].message.content
    history.append({"role": "assistant", "content": response})
    if len(history) > 5:
        history = history[-5:]
    return response, history

def transcribe_audio(audio):
    """Transcribes audio using Groq's Whisper model and returns the text."""
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)

    try:
        # Save the audio array as a temporary WAV file
        temp_filename = "temp_audio.wav"
        wavfile.write(temp_filename, rate=audio[0], data=audio[1])

        with open(temp_filename, "rb") as f:
            audio_content = f.read()

        translation = client.audio.translations.create(
            file=(temp_filename, audio_content), model="whisper-large-v3"
        )

        # Remove the temporary file
        os.remove(temp_filename)

        return translation.text
    except Exception as e:
        print(f"Error during transcription: {e}")
        return None

def convert_text_to_speech(text):
    """Convert text to speech using gTTS."""
    tts = gTTS(text, lang='en')
    audio_file = "output.mp3"
    tts.save(audio_file)
    return audio_file

# New function for real-time audio recording
def record_audio(audio_queue, should_stop):
    CHUNK = 1024
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000

    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT,
                    channels=CHANNELS,
                    rate=RATE,
                    input=True,
                    frames_per_buffer=CHUNK)

    while not should_stop.is_set():
        data = stream.read(CHUNK)
        audio_queue.put(data)

    stream.stop_stream()
    stream.close()
    p.terminate()




def create_audio_queue():
    return queue.Queue()

def create_stop_event():
    return threading.Event()

def process_input(input_type, text_input, audio_input, history, audio_queue_placeholder, should_stop_placeholder):
    audio_queue = create_audio_queue()
    should_stop = create_stop_event()

    # Start the audio recording thread
    audio_thread = threading.Thread(target=record_audio, args=(audio_queue, should_stop))
    audio_thread.start()

    if input_type == "text":
        response, new_history = reply(text_input, history)
    elif input_type == "audio":
        if audio_input is not None:
            transcribed_text = transcribe_audio(audio_input)
        else:
            # Process real-time audio
            audio_data = b''.join(list(audio_queue.queue))
            audio_queue.queue.clear()

            if audio_data:
                with wave.open("temp_realtime_audio.wav", "wb") as wf:
                    wf.setnchannels(1)
                    wf.setsampwidth(2)
                    wf.setframerate(16000)
                    wf.writeframes(audio_data)

                transcribed_text = transcribe_audio(("temp_realtime_audio.wav", 16000))
                os.remove("temp_realtime_audio.wav")
            else:
                transcribed_text = None

        if transcribed_text:
            response, new_history = reply(transcribed_text, history)
        else:
            response = "No speech detected or failed to transcribe audio."
            new_history = history
    else:
        response = "Invalid input type."
        new_history = history

    # Stop the audio recording thread
    should_stop.set()
    audio_thread.join()

    audio_response = convert_text_to_speech(response)
    return response, audio_response, new_history

def chat_interface(input_type, text_input, audio_input, history):
    audio_queue = create_audio_queue()
    should_stop = create_stop_event()

    response, audio_response, new_history = process_input(input_type, text_input, audio_input, history, audio_queue, should_stop)
    return response, audio_response, new_history


# Define the Gradio interface
# Define the Gradio interface
iface = gr.Interface(
    fn=chat_interface,
    inputs=[
        gr.Radio(["text", "audio"], label="Input Type"),
        gr.Textbox(label="Text Input"),
        gr.Audio(type="numpy", label="Audio Input (Optional)"),
        gr.State([]),  # For maintaining conversation history
    ],
    outputs=[
        gr.Textbox(label="Response"),
        gr.Audio(label="Audio Response"),
        gr.State()  # For updating conversation history
    ],
    title="Groq Chat Interface with Real-time Audio Support",
    description="Chat with the Groq AI model using text or real-time audio input."
)


# Launch the interface
iface.launch()
# Launch the interface


In [ ]:
!pip install speech_recognition

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
!apt-get update
!apt-get install python3.8

In [ ]:
!apt-get update
!apt-get install python3.9

In [ ]:
!ls /usr/bin/python*

In [ ]:
!apt install python3.8-distutils

In [ ]:
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.8 2
!update-alternatives --install /usr/local/bin/python3 python3 /usr/bin/python3.9 1
!python --version
!apt-get update
!apt install software-properties-common
!sudo dpkg --remove --force-remove-reinstreq python3-pip python3-setuptools python3-wheel
!apt-get install python3-pip

print('Git clone project and install requirements...')
!git clone https://github.com/Winfredy/SadTalker &> /dev/null
%cd SadTalker
!export PYTHONPATH=/content/SadTalker:$PYTHONPATH
!python3.8 -m pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!apt update
!apt install ffmpeg &> /dev/null
!python3.8 -m pip install -r requirements.txt

In [ ]:
print('Download pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

In [ ]:
# borrow from makeittalk
import ipywidgets as widgets
import glob
import matplotlib.pyplot as plt
print("Choose the image name to animate: (saved in folder 'examples/')")
img_list = glob.glob1('examples/source_image', '*.png')
img_list.sort()
img_list = [item.split('.')[0] for item in img_list]
default_head_name = widgets.Dropdown(options=img_list, value='full3')
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        plt.imshow(plt.imread('examples/source_image/{}.png'.format(default_head_name.value)))
        plt.axis('off')
        plt.show()
default_head_name.observe(on_change)
display(default_head_name)
plt.imshow(plt.imread('examples/source_image/{}.png'.format(default_head_name.value)))
plt.axis('off')
plt.show()

In [ ]:
# selected audio from exmaple/driven_audio
img = 'examples/source_image/{}.png'.format(default_head_name.value)
print(img)
!python3.8 inference.py --driven_audio ./examples/driven_audio/output.wav \
           --source_image {img} \
           --result_dir ./results  --preprocess crop --enhancer gfpgan

In [ ]:
# visualize code from makeittalk
from IPython.display import HTML
from base64 import b64encode
import os, sys

# get the last from results

results = sorted(os.listdir('./results/'))

mp4_name = glob.glob('./results/2024_08_20_20.12.40.mp4')[0]

mp4 = open('{}'.format(mp4_name),'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

print('Display animation: {}'.format(mp4_name), file=sys.stderr)
display(HTML("""
  <video width=256 controls>
        <source src="%s" type="video/mp4">
  </video>
  """ % data_url))


# GROG

In [ ]:
2+3

In [ ]:
!pip install -q -U groq

In [ ]:
import os
from groq import Groq
Groq_Api_Key = os.environ["GROQ_API_KEY"]
client = Groq(api_key=Groq_Api_Key)
chat_complete=client.chat.completions.create(
messages=[
    {
        'role':'user',
        'content':'Explain the importance of fast language model',

    }

],
model='llama-3.1-70b-versatile',
max_tokens=1024,
temperature=0.5,
stop=None,
stream=None



)
print(chat_complete.choices[0].message.content)

In [ ]:
chat_complete=client.chat.completions.create(
messages=[
    {
        'role':'user',
        'content':'What is my previous prompt',

    }

],
model='llama-versatile',
max_tokens=1024,
temperature=0.5,
stop=None,
stream=None



)
print(chat_complete.choices[0].message.content)

In [ ]:
import os
from groq import Groq


def audio(filepath):

  Groq_Api_Key = os.environ["GROQ_API_KEY"]
  client = Groq(api_key=Groq_Api_Key)
  with open(filepath,'rb') as file:
    translation = client.audio.translations.create(
        file=(filepath,file.read()),
        model='whisper-large-v3',
    )

    return translation.text




In [ ]:


audio('HH.mp3')

In [ ]:
filepath='HH.mp3'
translation_text = audio(filepath)

translation_text

In [ ]:

import gradio as gr


iface = gr.Interface(
    fn=audio,
    inputs="audio",
    outputs="text",
    title="Voice to Text"
)

iface.launch()





In [ ]:
import os


def reply(prompt, history=[]):
    Groq_Api_Key = os.environ["GROQ_API_KEY"]
    client = Groq(api_key=Groq_Api_Key)
    history.append({"role": "user", "content": prompt})


    chat_complete = client.chat.completions.create(
        messages=history,
        model='llama-3.1-70b-versatile',
        max_tokens=1024,
        temperature=0.5,
        stop=None,
        stream=None
    )

    history.append({"role": "assistant", "content": chat_complete.choices[0].message.content})
    if len(history) > 5:
        history=history[-5:]
    return chat_complete.choices[0].message.content


history=[]
print(reply('I have two dogs',history))

In [ ]:
print(reply('Are you sure',history))

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

In [ ]:
iface = gr.Interface(
    fn=reply,
    inputs="text",
    outputs="text",
    title="Groq Chat",
    description="Ask a question and get a response using Groq or GenerativeAI (choose your model)",
)

In [ ]:
iface.launch()


# Hugging Face

In [ ]:
pip install langchain langchain_community


In [ ]:

import os
from langchain.llms import HuggingFaceHub
# set HUGGINGFACEHUB_API_TOKEN in your environment before running
HuggingFace_llm=HuggingFaceHub(repo_id='google/flan-t5-base')
HuggingFace_llm('')

In [ ]:
import os
from fastapi import FastAPI, HTTPException
from langchain.llms import HuggingFaceHub

app = FastAPI()

# set HUGGINGFACEHUB_API_TOKEN in your environment before running
huggingface_llm = HuggingFaceHub(repo_id='google/flan-t5-base')

@app.post("/generate_text")
async def generate_text(prompt: str):
    try:
        response = huggingface_llm(prompt)
        return {"response": response}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
import requests

api_url = "http://localhost:8000/generate_text"  # Replace with your actual URL
prompt = "Write a poem about a cat chasing a butterfly."

data = {"prompt": prompt}
response = requests.post(api_url, json=data)

if response.status_code == 200:
  response_data = response.json()
  generated_text = response_data["response"]
  print(generated_text)
else:
  print(f"Error: {response.status_code}")


In [ ]:
langchain.